# Ambulance Dispatch Delay Risk Exploration

This notebook provides a compact exploration workflow for the synthetic ambulance dispatch dataset and the saved model results.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from preprocessing import AmbulanceDataPipeline

DATA_PATH = REPO_ROOT / 'data' / 'emergency_response_data.csv'
REPORT_PATH = REPO_ROOT / 'reports' / 'model_comparison.json'

df = pd.read_csv(DATA_PATH)
df.head()

## Dataset Overview

In [ ]:
df.info()
df.describe(include='all')

In [ ]:
df['delay_risk'].value_counts(normalize=True).rename('share')

## Preprocessing Shapes

In [ ]:
pipeline = AmbulanceDataPipeline()
train_df, val_df, test_df = pipeline.split_dataset(df)
pipeline.fit(train_df)

X_train_linear, y_train = pipeline.transform_linear(train_df)
X_train_integer, _ = pipeline.transform_integer(train_df)

{
    'train_rows': len(train_df),
    'validation_rows': len(val_df),
    'test_rows': len(test_df),
    'linear_feature_count': X_train_linear.shape[1],
    'integer_feature_count': X_train_integer.shape[1],
    'categorical_dims': pipeline.categorical_dims,
}

## Saved Model Results

In [ ]:
with open(REPORT_PATH, 'r', encoding='utf-8') as f:
    report = json.load(f)

pd.DataFrame(report['model_performance']).T

## Retraining

Run the full pipeline from the repository root:

```bash
/home/priya_paul/.venv/bin/python src/main.py
```